# Data Preprocessing

This notebook demonstrates loading the WikiQA and TREC-QA datasets, and running our preprocessing pipeline.

In [ ]:
import sys
sys.path.append('..')

from datasets import load_dataset
from utils.data_utils import *
import torch
from torch.utils.data import DataLoader

# Environment setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

### 1. Load Datasets
We load WikiQA and TREC-QA from HuggingFace.

In [ ]:
wiki_qa = load_dataset('wiki_qa')
trec_qa_raw = load_dataset('lucadiliello/trecqa')

print('WikiQA splits:', list(wiki_qa.keys()))
print('TREC-QA splits:', list(trec_qa_raw.keys()))

### 2. Build Vocabulary & Embeddings
We build a vocabulary from the training sets of both datasets, then load GloVe embeddings.

In [ ]:
word2idx, idx2word = build_vocab(wiki_qa['train'], trec_qa_raw['train'])
print(f'Vocabulary size: {len(word2idx)}')

download_glove()
glove_vectors = load_glove()
embedding_matrix = build_embedding_matrix(word2idx, glove_vectors)

### 3. Create PyTorch Datasets
We instantiate the `QADataset` which handles tokenization, padding, and finding question word positions in the answers.

In [ ]:
BATCH_SIZE = 64

# WikiQA
wiki_train_ds = QADataset(wiki_qa['train'], word2idx)
wiki_train_loader = DataLoader(wiki_train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

print(f'WikiQA Train size: {len(wiki_train_ds)}')

# Display sample
sample = wiki_train_ds[0]
print('\nSample Keys:', sample.keys())
print('Question Length:', sample['q_len'])
print('Answer Length:', sample['a_len'])
print('Label:', sample['label'].item())